# Episode 1 — Binance data with CCXT

A small, readable pipeline in four steps: **connect, pull, clean, indicators.** It uses [CCXT](https://github.com/ccxt/ccxt) so the exchange is swappable in one line, and writes a CSV so every later run works from the same data.

Run top to bottom. No API keys are needed: fetching candlesticks is a public endpoint.

In [1]:
import ccxt, pandas as pd, numpy as np, time

PAIR, TIMEFRAME, MONTHS = "BTC/USDT", "1h", 6

## 1. Connect

One exchange object. To move to another venue later you change this single line (`ccxt.binance()` → `ccxt.kraken()`); the rest of the notebook is unchanged.

In [2]:
ex = ccxt.binance()

## 2. Pull six months of hourly candles

Binance returns at most 1000 bars per call, so we page from a start time until we have the whole window, then save the raw pull as a CSV.

In [3]:
since = ex.parse8601((pd.Timestamp.utcnow() - pd.DateOffset(months=MONTHS)).strftime("%Y-%m-%dT%H:%M:%SZ"))
rows, ms_hour = [], 3600_000
while True:
    batch = ex.fetch_ohlcv(PAIR, TIMEFRAME, since=since, limit=1000)
    if not batch:
        break
    rows += batch
    since = batch[-1][0] + ms_hour
    if len(batch) < 1000:
        break
    time.sleep(ex.rateLimit / 1000)

df = pd.DataFrame(rows, columns=["time", "open", "high", "low", "close", "volume"])
df["time"] = pd.to_datetime(df["time"], unit="ms")
df = df.drop_duplicates("time").set_index("time").sort_index()
df.to_csv("data/BTCUSDT_1h_raw.csv")
print(f"{len(df)} rows  {df.index.min()} -> {df.index.max()}")
df.tail()

4368 rows  2025-12-18 12:00:00 -> 2026-06-18 11:00:00


,open,high,low,close,volume
time,,,,,
2026-06-18 07:00:00,64084.00,64510.50,64015.42,64510.44,524.85633
2026-06-18 08:00:00,64510.45,64646.75,64378.25,64399.92,714.20438
2026-06-18 09:00:00,64399.93,64478.00,64196.14,64199.53,300.31875
2026-06-18 10:00:00,64199.53,64223.79,63910.00,64078.61,445.57306
2026-06-18 11:00:00,64078.61,64196.00,63878.14,63906.00,326.91489


## 3. Clean

Two known Binance pain points: occasional missing hours, and zero-volume bars on quiet periods that break later math. We reindex onto a complete hourly grid and forward-fill, then replace any zero volume with a tiny number.

In [4]:
full = pd.date_range(df.index.min(), df.index.max(), freq="h")
gaps = len(full) - len(df)
df = df.reindex(full).ffill()
zeros = int((df["volume"] == 0).sum())
df.loc[df["volume"] == 0, "volume"] = 1e-10
df.index.name = "time"
print(f"filled {gaps} missing hours, fixed {zeros} zero-volume bars")

filled 0 missing hours, fixed 0 zero-volume bars


## 4. Indicators

Three classics on the close: **RSI(14)**, **MACD(12,26,9)**, **Bollinger(20,2)**. They are written out in pandas so each line is legible. To use the blog's TA-Lib instead, swap each block for the one-liner, e.g. `df['rsi14'] = talib.RSI(c, 14)`.

In [5]:
c = df["close"]
delta = c.diff()
gain = delta.clip(lower=0).ewm(alpha=1/14, adjust=False).mean()
loss = (-delta.clip(upper=0)).ewm(alpha=1/14, adjust=False).mean()
df["rsi14"] = 100 - 100 / (1 + gain / loss)

ema12, ema26 = c.ewm(span=12, adjust=False).mean(), c.ewm(span=26, adjust=False).mean()
df["macd"] = ema12 - ema26
df["macd_signal"] = df["macd"].ewm(span=9, adjust=False).mean()

mid, sd = c.rolling(20).mean(), c.rolling(20).std()
df["bb_mid"], df["bb_up"], df["bb_low"] = mid, mid + 2*sd, mid - 2*sd

df.to_csv("data/BTCUSDT_1h.csv")
df[["close","volume","rsi14","macd","macd_signal","bb_up","bb_low"]].tail()

,close,volume,rsi14,macd,macd_signal,bb_up,bb_low
time,,,,,,,
2026-06-18 07:00:00,64510.44,524.85633,45.345109,-353.464723,-340.995041,66053.599579,63438.425421
2026-06-18 08:00:00,64399.92,714.20438,43.800454,-329.519982,-338.700029,66022.540709,63405.388291
2026-06-18 09:00:00,64199.53,300.31875,41.068756,-322.990171,-335.558058,65959.605095,63360.476905
2026-06-18 10:00:00,64078.61,445.57306,39.469131,-323.839462,-333.214338,65915.805955,63301.992045
2026-06-18 11:00:00,63906.00,326.91489,37.239354,-334.583841,-333.488239,65741.776006,63291.421994


---
The cleaned, indicator-enriched series is saved to `data/BTCUSDT_1h.csv`. Episode 2 selects indicators per trading scenario.